# Training of the 55-thousand-case model

MBA in AI and Big Data final project — DeepONet for stress fields.

**Before running:** menu `Runtime` → `Change runtime type` → **GPU**.
Without it training falls back to the CPU and takes ten times longer.

The notebook was written to **survive session drops**: the training state
lives on Drive and the training cell can be re-run as many times as
needed, resuming where it stopped.

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
print('\nCUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('memory: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory/1e9))
else:
    print('NO GPU — change the runtime type before continuing')

## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/tcc'   # adjust if you used another folder
assert os.path.isdir(DRIVE), f'{DRIVE} does not exist'
print(os.listdir(DRIVE))

## 3. Extract to the LOCAL disk

Reading 55 thousand small files straight from Drive is extremely slow —
every read becomes a network call. Copying the `.tar` and extracting it on
the virtual machine's disk solves it. It takes a few minutes, and they are
well spent.

In [ ]:
%%time
!mkdir -p /content/work && cd /content/work && \
  cp {DRIVE}/codigo.tar . && tar -xf codigo.tar && \
  cp {DRIVE}/dataset_v4.tar . && \
  mkdir -p dataset_v4_treino && tar -xf dataset_v4.tar -C dataset_v4_treino && \
  rm -f dataset_v4.tar codigo.tar
!ls /content/work
!ls /content/work/dataset_v4_treino/samples | wc -l
!du -sh /content/work/dataset_v4_treino

## 4. Quick test before committing hours of GPU

Three epochs. It catches import, path or dimension errors now, and it
measures the time per epoch — which is what allows the total cost to be
estimated.

In [ ]:
%%time
%cd /content/work
!python train_deeponet.py --dataset dataset_v4_treino --device cuda \
  --epochs 3 --eval-every 1 --hole-distances --decoder prod \
  --stats-cache /content/stats_55k.pkl --final-eval 400 \
  --out /content/teste.pt

## 5. Training

The checkpoint is written **to Drive**, to survive the end of the session.
If the session drops, just re-run this cell: `--resume` picks up from the
epoch where it stopped, with the optimizer and scheduler state.

`MODEL` selects the configuration:

- `'delivered'` — the model delivered with the monograph: fusion decoder and
  hole distances, plain MSE. Start with 300 epochs, look at the curve, and
  increase `EPOCHS` by re-running the cell if the validation error is still
  going down. (The delivered model was later resumed with `--lr 3e-4`; set
  `LR = 3e-4` for that one run, which restarts the rate and the scheduler.)
- `'revised'` — the revised model: adds the concave-edge distance and the
  hole proximity to the trunk, and weights the loss by stress
  (`--stress-weight 1 --weight-floor 0.3`). 4000 epochs at `--lr 3e-4`, the
  learning rate at which the delivered model had been restarted; `--lr` is
  passed only at the first start, so a resume keeps the scheduler state. It
  needs `concave_edges.pkl` in the dataset folder
  (`python concave_edges.py --dataset ...`): run the next cell once first.
  `codigo.tar` must contain the current scripts, `concave_edges.py`
  included.

Each configuration writes its own checkpoint and log, so switching `MODEL`
does not mix them.

### Concave edges (revised model only)

`--edge-distance` reads `concave_edges.pkl` from the dataset folder. Build it
once (it reads every case); copy it to Drive to avoid rebuilding it in a new
session. A `fillet_edges.pkl` from an earlier run is also accepted.

In [ ]:
%cd /content/work
import os
if not any(os.path.exists(f'dataset_v4_treino/{n}')
           for n in ('concave_edges.pkl', 'fillet_edges.pkl')):
    if os.path.exists(f'{DRIVE}/concave_edges.pkl'):
        !cp '{DRIVE}/concave_edges.pkl' dataset_v4_treino/
    else:
        !python concave_edges.py --dataset dataset_v4_treino
        !cp dataset_v4_treino/concave_edges.pkl '{DRIVE}/'
!ls -la dataset_v4_treino/*.pkl

In [ ]:
MODEL = 'delivered'   # 'delivered' or 'revised'

STATS = '/content/stats_55k.pkl'   # local cache; recomputed if the session changes
if MODEL == 'delivered':
    EPOCHS = 300
    # None = 1e-3 at the first start, and on a resume the checkpoint's rate.
    # Set 3e-4 for ONE run to restart the rate and the scheduler, as was done
    # for the delivered model, then set it back to None.
    LR = None
    FLAGS = '--hole-distances --decoder prod'
    CKPT = f'{DRIVE}/deeponet_6fam_55k.pt'
    LOG  = f'{DRIVE}/treino_55k.log'
elif MODEL == 'revised':
    EPOCHS = 4000
    CKPT = f'{DRIVE}/deeponet_6fam_55k_revised.pt'
    LOG  = f'{DRIVE}/train_55k_revised.log'
    # initial rate only: passing --lr on a resume would restart the rate and
    # the scheduler every time the session drops
    LR = None if os.path.exists(CKPT) else 3e-4
    FLAGS = ('--decoder prod --hole-distances --edge-distance '
             '--hole-proximity --stress-weight 1 --weight-floor 0.3')
else:
    raise ValueError(MODEL)
LR_OPT = '' if LR is None else f'--lr {LR}'

%cd /content/work
!python -u train_deeponet.py --dataset dataset_v4_treino --device cuda \
  --epochs {EPOCHS} --batch 32 --n-query 1024 --repeats 1 {LR_OPT} \
  {FLAGS} --eval-every 10 --save-every 10 \
  --stats-cache '{STATS}' --final-eval 400 \
  --resume --out '{CKPT}' 2>&1 | tee -a '{LOG}'

## 6. Follow the curve

In [ ]:
import re, matplotlib.pyplot as plt
ep, tr, va = [], [], []
for ln in open(LOG, errors='ignore'):
    # current log format ('epoch ... | relL2 train mean=... | VAL mean=...') and
    # the one written before the translation ('época ... treino méd=... VAL méd=...')
    m = re.search(r'(?:epoch|época)\s+(\d+).*?(?:train mean|treino méd)=([\d.]+)'
                  r'.*?VAL (?:mean|méd)=([\d.]+)', ln)
    if m:
        ep.append(int(m.group(1))); tr.append(float(m.group(2)))
        va.append(float(m.group(3)))
if ep:
    plt.figure(figsize=(8,4))
    plt.plot(ep, tr, label='training'); plt.plot(ep, va, label='validation')
    plt.xlabel('epoch'); plt.ylabel('relL2'); plt.legend(); plt.grid(alpha=.3)
    plt.show()
    print(f'latest: training {tr[-1]:.3f} | val {va[-1]:.3f} | gap {va[-1]-tr[-1]:.3f}')
    if len(va) > 5:
        d = va[-5] - va[-1]
        print(f'drop over the last 5 evaluations: {d:+.4f}')
        print('still going down — worth increasing EPOCHS' if d > 0.002
              else 'stabilized — you can stop')
else:
    print('no evaluation line in the log yet')

## 7. Download the model

The checkpoint is already on Drive. This cell only downloads it straight
to the computer, if you prefer.

In [ ]:
import torch
ck = torch.load(CKPT, map_location='cpu', weights_only=False)
print('epoch:', ck.get('epoch'), '| best val MSE:', ck.get('best'))
# split key 'train'; checkpoints written before the translation use 'treino'
train_split = ck['split'].get('train', ck['split'].get('treino', []))
print('training cases:', len(train_split),
      '| val:', len(ck['split']['val']))
print('branch_dim:', ck['stats']['branch_dim'])

from google.colab import files
files.download(CKPT)